# Probe T4 + SOAR (2 min)


In [ ]:

import os, time, traceback
print("PROBE start", flush=True)
import torch
print("torch", torch.__version__, "cuda", torch.cuda.is_available(), flush=True)
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0), flush=True)
MODEL_DIRS=[]
for r,_,fs in os.walk("/kaggle/input"):
    if "config.json" in fs and ("soar" in r.lower() or "qwen" in r.lower()):
        if any(f.endswith(".safetensors") or f.endswith(".bin") for f in fs):
            MODEL_DIRS.append(r)
print("models:", MODEL_DIRS, flush=True)
assert MODEL_DIRS, "no model found"
from transformers import AutoTokenizer, AutoModelForCausalLM
md=sorted(MODEL_DIRS,key=len)[0]
print("loading", md, flush=True)
t0=time.time()
tok=AutoTokenizer.from_pretrained(md, trust_remote_code=True)
if tok.pad_token is None: tok.pad_token=tok.eos_token
model=AutoModelForCausalLM.from_pretrained(md, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True, low_cpu_mem_usage=True)
model.eval()
print(f"loaded in {time.time()-t0:.1f}s device={next(model.parameters()).device}", flush=True)
ids=tok("def solve(grid):\n return grid", return_tensors="pt")
ids={k:v.to(model.device) for k,v in ids.items()}
t1=time.time()
with torch.no_grad():
    out=model.generate(**ids, max_new_tokens=32, do_sample=False)
print(f"gen 32 tok in {time.time()-t1:.1f}s", flush=True)
print(tok.decode(out[0], skip_special_tokens=True)[-200:], flush=True)
print("PROBE OK", flush=True)
